In [0]:
USE CATALOG workspace;
USE SCHEMA bde;


In [0]:
SELECT COUNT(*) AS trips FROM workspace.bde.taxi_trips_final;


trips
967861897


In [0]:
SELECT COUNT(*) AS baseline_rows FROM baseline_q3c;


baseline_rows
1058846


In [0]:
SELECT SUM(CASE WHEN yy=2024 AND mm IN (10,11,12) THEN 1 ELSE 0 END) AS q4_in_train
FROM ml_train_tiny;

SELECT SUM(CASE WHEN NOT (yy=2024 AND mm IN (10,11,12)) THEN 1 ELSE 0 END) AS non_q4_in_test
FROM ml_test_q4_tiny;

-- OPTIONAL: sizes
SELECT 'train' AS which, COUNT(*) AS rows FROM ml_train_tiny
UNION ALL
SELECT 'test'  AS which, COUNT(*) FROM ml_test_q4_tiny;



which,rows
train,957249
test,21448


In [0]:
SELECT
  COUNT(*) > 0 AS has_forbidden_cols
FROM (
  SELECT 1
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE table_catalog='workspace'
    AND table_schema='bde'
    AND table_name='ml_train_tiny'
    AND column_name IN ('fare_amount','tolls_amount')
) t;


has_forbidden_cols
false


In [0]:
WITH m AS (
  SELECT
    service_type, pickup_borough, dropoff_borough,
    date_format(to_timestamp(concat(yy,'-',lpad(mm,2,'0'),'-01')),'yyyy-MM') AS yyyymm,
    dow, hr, total_amount
  FROM ml_train_tiny
),
j AS (
  SELECT b.avg_total_amount AS pred, m.total_amount AS label
  FROM m JOIN baseline_q3c b
    USING (service_type, pickup_borough, dropoff_borough, yyyymm, dow, hr)
)
SELECT COUNT(*) AS joined_rows,
       SQRT(AVG(POWER(j.label - j.pred,2))) AS baseline_rmse_train
FROM j;


joined_rows,baseline_rmse_train
957249,32.20703070787696


In [0]:
WITH m AS (
  SELECT
    service_type, pickup_borough, dropoff_borough,
    date_format(to_timestamp(concat(yy,'-',lpad(mm,2,'0'),'-01')),'yyyy-MM') AS yyyymm,
    dow, hr, total_amount
  FROM ml_test_q4_tiny
),
j AS (
  SELECT b.avg_total_amount AS pred, m.total_amount AS label
  FROM m JOIN baseline_q3c b
    USING (service_type, pickup_borough, dropoff_borough, yyyymm, dow, hr)
)
SELECT COUNT(*) AS joined_rows,
       SQRT(AVG(POWER(j.label - j.pred,2))) AS baseline_rmse_test
FROM j;


joined_rows,baseline_rmse_test
21448,11.742405288736142


In [0]:
%python
import pandas as pd, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# tiny train sample (already created in SQL)
train_pd = spark.table("workspace.bde.ml_train_tiny").toPandas()

cat = ["service_type","pickup_borough","dropoff_borough","dow","ratecodeid","payment_type","vendorid"]
num = ["duration_min","distance_km","passenger_count","hr","mm","yy"]
for c in cat: train_pd[c] = train_pd[c].astype("string")

X = train_pd[cat+num]
y = train_pd["total_amount"].astype(float)

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=7)

# DENSE OHE so HistGBR works; simple imputers; no scaler (keeps memory small)
prep = ColumnTransformer([
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat),
    ("num", SimpleImputer(strategy="median"), num)
])

models = {
    "Ridge":   Pipeline([("prep", prep), ("m", Ridge(alpha=1.0, random_state=7))]),
    "HistGBR": Pipeline([("prep", prep), ("m", HistGradientBoostingRegressor(random_state=7))]),
}

rows=[]; best_name=None; best_pipe=None; best_val=float("inf")
for name, pipe in models.items():
    t0=time.perf_counter(); pipe.fit(X_tr, y_tr); fit_s=time.perf_counter()-t0
    val = rmse(y_va, pipe.predict(X_va))
    rows.append([name, round(val,3), round(fit_s,2)])
    if val < best_val: best_name, best_pipe, best_val = name, pipe, val

display(pd.DataFrame(rows, columns=["model","rmse_valid","fit_time_sec"]).sort_values("rmse_valid"))
print("Selected best model:", best_name)


model,rmse_valid,fit_time_sec
HistGBR,2.922,17.52
Ridge,4.274,5.94


Selected best model: HistGBR


In [0]:
%python
from sklearn.metrics import root_mean_squared_error as rmse


train_pd = spark.table("workspace.bde.ml_train_tiny").toPandas()
for c in ["service_type","pickup_borough","dropoff_borough","dow","ratecodeid","payment_type","vendorid"]:
    train_pd[c] = train_pd[c].astype("string")
X_all = train_pd[["service_type","pickup_borough","dropoff_borough","dow",
                  "ratecodeid","payment_type","vendorid",
                  "duration_min","distance_km","passenger_count","hr","mm","yy"]]
y_all = train_pd["total_amount"].astype(float)
best_pipe.fit(X_all, y_all)

# TEST (Q4-2024)
test_pd = spark.table("workspace.bde.ml_test_q4_tiny").toPandas()
for c in ["service_type","pickup_borough","dropoff_borough","dow","ratecodeid","payment_type","vendorid"]:
    test_pd[c] = test_pd[c].astype("string")
X_test = test_pd[["service_type","pickup_borough","dropoff_borough","dow",
                  "ratecodeid","payment_type","vendorid",
                  "duration_min","distance_km","passenger_count","hr","mm","yy"]]
y_test = test_pd["total_amount"].astype(float)

rmse_test_model = rmse(y_test, best_pipe.predict(X_test))

# Baseline RMSE on the same test sample
baseline_rmse_test = spark.sql("""
WITH m AS (
  SELECT
    service_type, pickup_borough, dropoff_borough,
    date_format(to_timestamp(concat(yy,'-',lpad(mm,2,'0'),'-01')),'yyyy-MM') AS yyyymm,
    dow, hr, total_amount
  FROM workspace.bde.ml_test_q4_tiny
),
j AS (
  SELECT b.avg_total_amount AS pred, m.total_amount AS label
  FROM m JOIN workspace.bde.baseline_q3c b
    USING (service_type, pickup_borough, dropoff_borough, yyyymm, dow, hr)
)
SELECT SQRT(AVG(POWER(j.label - j.pred, 2))) AS rmse
FROM j
""").first()["rmse"]

print(f"{best_name} — RMSE on 2024-Q4 (sample): {rmse_test_model:,.3f}")
print(f"Baseline (Q3-c) RMSE on 2024-Q4  : {baseline_rmse_test:,.3f}")
print("Beats baseline?                  :", "YES" if rmse_test_model < baseline_rmse_test else "NO")


HistGBR — RMSE on 2024-Q4 (sample): 4.910
Baseline (Q3-c) RMSE on 2024-Q4  : 11.742
Beats baseline?                  : YES


In [0]:
%python
from sklearn.metrics import root_mean_squared_error as rmse

y_tr_pred = best_pipe.predict(X_tr)
y_va_pred = best_pipe.predict(X_va)

print(f"Train RMSE     : {rmse(y_tr, y_tr_pred):.3f}")
print(f"Validation RMSE: {rmse(y_va, y_va_pred):.3f}")


Train RMSE     : 2.519
Validation RMSE: 2.702
